# 05 · Contact trajectories (group-based trajectory models)

Fits group-based trajectory models with one to six classes to unique contact days per quarter in the landmark population, summarizes classification quality, names the classes of the selected solution, and saves each patient's class and posterior probabilities.

**Reads (`DATA_DIR`):** `quarterly_unique`, `landmark_build` (notebook 02)

**Produces (patient-level, `DATA_DIR`):** fitted models `gbtm/k1.rds … k6.rds`; `trajectory_classes` (selected solution) and `trajectory_classes_k4`, `trajectory_classes_k5`

**Produces (aggregate, `results/`):** `tableS2a_fit.csv`, `tableS2b_classes.csv` (selected), `tableS2b_classes_k5.csv` (alternative), `figure3a_trajectories.csv`, `05_trajectories.json`

**Paper items:** Methods, *Contact measures*; Supplementary Methods, *Trajectory models*; Table S2; Figure 3A; class sizes and posterior-probability results in the text.

**Model:** `hlme` (R package `lcmm`): outcome y(i,q) = log(1 + unique contact days) in quarters 1–7 (days 91–727), quadratic time in the class-specific mean, Gaussian errors, no random effects. For 2–6 classes, 20 random starts of 15 iterations from the one-class solution (`gridsearch`, seed 42).

**Runtime:** one to several hours, mostly the 5- and 6-class models. Each model is saved to `DATA_DIR` as soon as it finishes, and a rerun loads finished models instead of refitting them.

## 0 · Setup

In [ ]:
# ---- edit these ----
REPO_DIR = "/content/drive/MyDrive/Ronveer Sandhu/Project 1 -MIMIC analysis-hypertension retention/htn-bp-followup-mimic"
DATA_DIR = "/content/drive/MyDrive/htn_bp_data"
K_SELECTED = 4      # selected solution: smallest k meeting the rule in section 3 (checked in section 4)
K_ALT      = 5      # alternative solution kept for the sensitivity analysis in notebook 06b

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, json
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from htn_utils import *

import numpy as np
import pandas as pd

RESULTS_DIR = os.path.join(REPO_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
RDS_DIR = os.path.join(DATA_DIR, "gbtm")
os.makedirs(RDS_DIR, exist_ok=True)

lb = load_df(DATA_DIR, "landmark_build")
qu = load_df(DATA_DIR, "quarterly_unique")

## 1 · Trajectory input

One row per landmark patient and quarter: U = unique contact days, y = log(1 + U).

In [ ]:
ids_lm = lb.loc[lb.in_landmark, "subject_id"].values
gin = (pd.MultiIndex.from_product([ids_lm, range(1, 8)], names=["subject_id", "q"]).to_frame(index=False)
         .merge(qu, on=["subject_id", "q"], how="left").fillna({"n_unique": 0}))
gin["y_unique"] = np.log1p(gin.n_unique)
print(f"landmark patients: {len(ids_lm):,} | rows: {len(gin):,}")
print("mean unique contact days by quarter:", gin.groupby("q").n_unique.mean().round(2).to_dict())

gin_r = gin[["subject_id", "q", "y_unique"]].copy()
gin_r["subject_id"] = gin_r.subject_id.astype(int)
gin_r["q"] = gin_r.q.astype(int)

## 2 · Fit the models (R, `lcmm`)

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
options(repos = c(CRAN = "https://cloud.r-project.org"))
if (!requireNamespace("lcmm", quietly = TRUE)) install.packages("lcmm", quiet = TRUE)
suppressMessages(library(lcmm))
cat("lcmm", as.character(packageVersion("lcmm")), "\n")

In [ ]:
%%R -i gin_r -i RDS_DIR
suppressMessages(library(lcmm))
d <- as.data.frame(gin_r)
t0 <- Sys.time()

f1 <- file.path(RDS_DIR, "k1.rds")
if (file.exists(f1)) {
  m1 <- readRDS(f1); cat("k=1 loaded\n")
} else {
  set.seed(42)
  m1 <- hlme(y_unique ~ q + I(q^2), subject = "subject_id", ng = 1, data = d, verbose = FALSE)
  saveRDS(m1, f1)
  cat(sprintf("k=1  loglik=%.1f  BIC=%.1f\n", m1$loglik, m1$BIC))
}

for (k in 2:6) {
  fk <- file.path(RDS_DIR, sprintf("k%d.rds", k))
  if (file.exists(fk)) { cat(sprintf("k=%d loaded\n", k)); next }
  set.seed(42)
  mk <- gridsearch(rep = 20, maxiter = 15, minit = m1,
                   hlme(y_unique ~ q + I(q^2), mixture = ~ q + I(q^2),
                        subject = "subject_id", ng = k, data = d, verbose = FALSE))
  saveRDS(mk, fk)
  cat(sprintf("k=%d  conv=%d  loglik=%.1f  BIC=%.1f  [%.1f min]\n",
              k, mk$conv, mk$loglik, mk$BIC, as.numeric(difftime(Sys.time(), t0, units = "mins"))))
}

In [ ]:
%%R -i RDS_DIR
# fit statistics and posterior probabilities for Python
stats <- data.frame()
for (k in 1:6) {
  m <- readRDS(file.path(RDS_DIR, sprintf("k%d.rds", k)))
  stats <- rbind(stats, data.frame(k = k, conv = m$conv, npm = length(m$best),
                                   loglik = m$loglik, AIC = m$AIC, BIC = m$BIC))
  if (k > 1) write.csv(m$pprob, file.path(RDS_DIR, sprintf("pprob_k%d.csv", k)), row.names = FALSE)
}
write.csv(stats, file.path(RDS_DIR, "fit_stats.csv"), row.names = FALSE)

## 3 · Classification quality (Table S2A)

For each solution: relative entropy, smallest class, lowest class-average posterior probability, lowest odds of correct classification, and the share of patients whose highest posterior probability is below 0.70.

**Selection rule:** the smallest solution that separates rising from declining contact in which every class has an average posterior probability above 0.70, odds of correct classification above 5, and at least 2% of patients. The column `meets_class_criteria` checks the three numeric criteria; whether a solution separates rising from declining contact is read from the class shapes in section 4.

In [ ]:
stats = pd.read_csv(os.path.join(RDS_DIR, "fit_stats.csv"))

def class_quality(k):
    pp = pd.read_csv(os.path.join(RDS_DIR, f"pprob_k{k}.csv"))
    probs = pp[[f"prob{g}" for g in range(1, k + 1)]].values
    n = len(pp)
    ent = 1 + (probs * np.log(np.clip(probs, 1e-300, 1))).sum() / (n * np.log(k))
    rows = []
    for g in range(1, k + 1):
        sub = pp[pp["class"] == g]
        pi = len(sub) / n
        avepp = sub[f"prob{g}"].mean()
        occ = (avepp / (1 - avepp)) / (pi / (1 - pi)) if avepp < 1 else np.inf
        rows.append({"class": g, "n": len(sub), "pct": 100 * pi, "avepp": avepp, "occ": occ})
    q = pd.DataFrame(rows)
    summ = dict(entropy=ent, smallest_pct=q.pct.min(), lowest_avepp=q.avepp.min(), lowest_occ=q.occ.min(),
                maxpost_lt_070_pct=100 * (probs.max(1) < 0.70).mean())
    summ["meets_class_criteria"] = bool((q.avepp > 0.70).all() and (q.occ > 5).all() and (q.pct >= 2).all())
    return pp, summ, q

qual = {k: class_quality(k)[1] for k in range(2, 7)}
tableS2a = stats.merge(pd.DataFrame(qual).T.rename_axis("k").reset_index(), on="k", how="left")
print(tableS2a.round(2).to_string(index=False))

## 4 · Name the classes (Table S2B)

Classes are named by rule, not by their arbitrary `lcmm` numbers: lowest mean contact → **Minimal Contact**. In the 4-class solution the highest → **Sustained Contact**; in the 5-class solution the highest → **Sustained High Contact** and the second highest → **Sustained Moderate Contact**. Of the remaining two classes, the one whose contact rises from quarter 1 to quarter 7 → **Rising Contact**, the other → **Declining Contact**.

The class shapes for the 4-, 5-, and 6-class solutions are printed first. A solution separates rising from declining contact when, apart from Minimal Contact and the highest class, it has one class whose contact rises and one whose contact falls. In the corrected landmark population the 4-class solution already does this and meets every criterion, so it is selected; the 5-class solution (which splits sustained contact into moderate and high) is kept as a sensitivity analysis.

In [ ]:
def shapes(k):
    pp, _, q = class_quality(k)
    traj = gin.merge(pp[["subject_id", "class"]], on="subject_id").groupby(["class", "q"]).n_unique.mean().unstack("q")
    q["mean_unique_days"] = q["class"].map(traj.mean(axis=1))
    q["change_q7_minus_q1"] = q["class"].map(traj[7] - traj[1])
    return pp, q

for k in [4, 5, 6]:
    print(f"--- {k} classes ---")
    print(shapes(k)[1].sort_values("mean_unique_days").round(2).to_string(index=False))

TOP = {4: ["Sustained Contact"], 5: ["Sustained Moderate Contact", "Sustained High Contact"]}

def name_classes(k):
    pp_, q_ = shapes(k)
    order = q_.sort_values("mean_unique_days")["class"].tolist()
    names = {order[0]: "Minimal Contact"}
    top = TOP[k]
    for c, nm in zip(order[len(order) - len(top):], top):
        names[c] = nm
    mid = q_[q_["class"].isin(order[1:len(order) - len(top)])].sort_values("change_q7_minus_q1")
    assert len(mid) == 2 and mid.change_q7_minus_q1.iloc[0] < 0 < mid.change_q7_minus_q1.iloc[-1], \
        f"{k}-class solution does not separate rising from declining contact"
    names[mid["class"].iloc[0]] = "Declining Contact"
    names[mid["class"].iloc[-1]] = "Rising Contact"
    q_["label"] = q_["class"].map(names)
    order_names = ["Minimal Contact", "Declining Contact", "Rising Contact"] + top
    return pp_, q_.set_index("label").loc[order_names].reset_index(), names, order_names

SOL = {k: name_classes(k) for k in (K_SELECTED, K_ALT)}
for k in (K_SELECTED, K_ALT):
    assert tableS2a.set_index("k").loc[k, "meets_class_criteria"], f"{k}-class solution fails the class criteria"
    print(f"\nTable S2B ({k} classes{', selected' if k == K_SELECTED else ', sensitivity analysis'}):")
    print(SOL[k][1][["label", "n", "pct", "mean_unique_days", "change_q7_minus_q1", "avepp", "occ"]].round(2).to_string(index=False))

pp, tableS2b, NAMES, ORDER = SOL[K_SELECTED]

## 5 · Class assignments and Figure 3A data

Each patient is assigned to the class with the highest posterior probability; the posterior probabilities are kept for the sensitivity analysis that weights each patient across classes (notebook 06b). Figure 3A: mean of log(1 + unique contact days) by class and quarter with 95% confidence intervals, back-transformed to days.

In [ ]:
def assignments(k):
    pp_, _, names, order_names = SOL[k]
    c = pp_.rename(columns={f"prob{g}": f"p_{names[g]}" for g in names})
    c["label"] = c["class"].map(names)
    c["max_post"] = c[[f"p_{n}" for n in order_names]].max(axis=1)
    return c

CLS = {k: assignments(k) for k in (K_SELECTED, K_ALT)}
cls = CLS[K_SELECTED]
print(f"class sizes ({K_SELECTED} classes):", cls.label.value_counts().reindex(ORDER).to_dict())
alt_tab = pd.crosstab(cls.label, CLS[K_ALT].set_index("subject_id").loc[cls.subject_id, "label"].values)
print(f"\n{K_SELECTED}-class (rows) vs {K_ALT}-class (columns) assignment:")
print(alt_tab.reindex(index=ORDER, columns=SOL[K_ALT][3]).fillna(0).astype(int).to_string())
print(f"maximum posterior < 0.70: {100 * (cls.max_post < 0.70).mean():.1f}%")

f3 = (gin.merge(cls[["subject_id", "label"]], on="subject_id")
         .groupby(["label", "q"]).y_unique.agg(["mean", "std", "count"]).reset_index())
se = f3["std"] / np.sqrt(f3["count"])
f3["days"], f3["lo"], f3["hi"] = np.expm1(f3["mean"]), np.expm1(f3["mean"] - 1.96 * se), np.expm1(f3["mean"] + 1.96 * se)
figure3a = f3[["label", "q", "days", "lo", "hi", "count"]]

## 6 · Save

In [ ]:
save_df(cls, DATA_DIR, "trajectory_classes")
for k in (K_SELECTED, K_ALT):
    save_df(CLS[k], DATA_DIR, f"trajectory_classes_k{k}")
tableS2a.to_csv(os.path.join(RESULTS_DIR, "tableS2a_fit.csv"), index=False)
tableS2b.to_csv(os.path.join(RESULTS_DIR, "tableS2b_classes.csv"), index=False)
SOL[K_ALT][1].to_csv(os.path.join(RESULTS_DIR, f"tableS2b_classes_k{K_ALT}.csv"), index=False)
figure3a.to_csv(os.path.join(RESULTS_DIR, "figure3a_trajectories.csv"), index=False)
save_json({"fit_population_n": int(len(ids_lm)), "k_selected": K_SELECTED, "class_order": ORDER,
           "class_numbers_to_names": {int(k_): v for k_, v in NAMES.items()},
           "alternative": {"k": K_ALT, "class_order": SOL[K_ALT][3],
                           "class_numbers_to_names": {int(k_): v for k_, v in SOL[K_ALT][2].items()},
                           "class_sizes": CLS[K_ALT].label.value_counts().reindex(SOL[K_ALT][3]).astype(int).to_dict()},
           "class_sizes": cls.label.value_counts().reindex(ORDER).astype(int).to_dict(),
           "maxpost_lt_070_pct": float(100 * (cls.max_post < 0.70).mean())},
          RESULTS_DIR, "05_trajectories")